In [1]:
import argparse, collections, collections.abc, copy, hashlib, itertools, gzip, json, os, os.path, re, string, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd
import af3io, pooled_ppi

In [2]:
predictions = pd.concat([
    af3io.predictions.read_summary_confidences('/cluster/project/beltrao/jjaenes/26.01_batch-infer/pooled-ppi-yeast/data/predictions/26.02/euler-std-a100-r3/alphafold3_predictions/000d99f7e4eba5e9ac87155974c36658f0058d61.zip'),
    af3io.predictions.read_summary_confidences('/cluster/project/beltrao/jjaenes/26.01_batch-infer/pooled-ppi-yeast/data/predictions/26.02/euler-std-a100-r3/alphafold3_predictions/001780d42f7d1b3ba3a36f86b65db60af37183d1.zip'),
], axis=0)
predictions

,name,seed,sample,ranking_score,fraction_disordered,has_clash,iptm,ptm,chain_iptm,chain_pair_iptm,chain_pair_pae_min,chain_ptm,summary_confidences_path,predictions_path
0,000d99f7e4eba5e9ac87155974c36658f0058d61,4,0,0.342871,0.08,0.0,0.29,0.36,"[0.1, 0.06, 0.11, 0.08, 0.09, 0.07, 0.07, 0.07...","[[0.65, 0.08, 0.13, 0.1, 0.11, 0.09, 0.09, 0.0...","[[0.76, 31.12, 30.73, 31.04, 31.21, 31.34, 31....","[0.65, 0.23, 0.47, 0.4, 0.81, 0.73, 0.15, 0.29...",000d99f7e4eba5e9ac87155974c36658f0058d61/seed-...,/cluster/project/beltrao/jjaenes/26.01_batch-i...
1,000d99f7e4eba5e9ac87155974c36658f0058d61,4,1,0.342090,0.07,0.0,0.29,0.36,"[0.1, 0.06, 0.11, 0.08, 0.09, 0.07, 0.07, 0.07...","[[0.65, 0.08, 0.13, 0.1, 0.11, 0.09, 0.09, 0.0...","[[0.76, 31.31, 30.09, 30.89, 31.12, 31.44, 31....","[0.65, 0.23, 0.48, 0.4, 0.81, 0.73, 0.15, 0.3,...",000d99f7e4eba5e9ac87155974c36658f0058d61/seed-...,/cluster/project/beltrao/jjaenes/26.01_batch-i...
2,000d99f7e4eba5e9ac87155974c36658f0058d61,4,2,0.347959,0.08,0.0,0.29,0.36,"[0.1, 0.06, 0.11, 0.08, 0.09, 0.07, 0.07, 0.07...","[[0.64, 0.08, 0.13, 0.1, 0.11, 0.09, 0.09, 0.0...","[[0.76, 31.13, 30.95, 30.92, 31.07, 31.38, 31....","[0.64, 0.16, 0.47, 0.41, 0.8, 0.74, 0.15, 0.29...",000d99f7e4eba5e9ac87155974c36658f0058d61/seed-...,/cluster/project/beltrao/jjaenes/26.01_batch-i...
3,000d99f7e4eba5e9ac87155974c36658f0058d61,4,3,0.346393,0.08,0.0,0.29,0.36,"[0.1, 0.06, 0.11, 0.08, 0.09, 0.07, 0.07, 0.07...","[[0.65, 0.08, 0.13, 0.1, 0.11, 0.09, 0.09, 0.0...","[[0.76, 31.46, 29.2, 30.78, 30.96, 31.35, 31.0...","[0.65, 0.24, 0.48, 0.41, 0.81, 0.73, 0.15, 0.2...",000d99f7e4eba5e9ac87155974c36658f0058d61/seed-...,/cluster/project/beltrao/jjaenes/26.01_batch-i...
4,000d99f7e4eba5e9ac87155974c36658f0058d61,4,4,0.356635,0.11,0.0,0.29,0.36,"[0.1, 0.06, 0.11, 0.08, 0.09, 0.07, 0.07, 0.06...","[[0.64, 0.08, 0.14, 0.1, 0.11, 0.09, 0.09, 0.0...","[[0.76, 31.22, 26.28, 29.79, 30.79, 31.34, 30....","[0.64, 0.19, 0.47, 0.4, 0.81, 0.73, 0.17, 0.29...",000d99f7e4eba5e9ac87155974c36658f0058d61/seed-...,/cluster/project/beltrao/jjaenes/26.01_batch-i...
0,001780d42f7d1b3ba3a36f86b65db60af37183d1,4,0,0.328513,0.05,0.0,0.29,0.36,"[0.09, 0.09, 0.08, 0.09, 0.07, 0.1, 0.07, 0.09...","[[0.78, 0.09, 0.07, 0.09, 0.08, 0.1, 0.07, 0.0...","[[0.76, 30.74, 31.43, 31.52, 30.8, 30.8, 31.19...","[0.78, 0.82, 0.41, 0.81, 0.76, 0.83, 0.51, 0.7...",001780d42f7d1b3ba3a36f86b65db60af37183d1/seed-...,/cluster/project/beltrao/jjaenes/26.01_batch-i...
1,001780d42f7d1b3ba3a36f86b65db60af37183d1,4,1,0.321466,0.05,0.0,0.28,0.35,"[0.09, 0.09, 0.08, 0.09, 0.07, 0.1, 0.07, 0.08...","[[0.79, 0.09, 0.07, 0.09, 0.08, 0.1, 0.07, 0.0...","[[0.76, 30.72, 31.4, 31.15, 30.61, 31.0, 31.33...","[0.79, 0.82, 0.42, 0.81, 0.76, 0.83, 0.51, 0.7...",001780d42f7d1b3ba3a36f86b65db60af37183d1/seed-...,/cluster/project/beltrao/jjaenes/26.01_batch-i...
2,001780d42f7d1b3ba3a36f86b65db60af37183d1,4,2,0.332140,0.06,0.0,0.29,0.36,"[0.09, 0.09, 0.08, 0.09, 0.07, 0.09, 0.07, 0.0...","[[0.78, 0.09, 0.08, 0.09, 0.08, 0.1, 0.07, 0.0...","[[0.76, 31.03, 31.28, 31.28, 30.7, 30.81, 31.1...","[0.78, 0.82, 0.41, 0.81, 0.76, 0.82, 0.51, 0.7...",001780d42f7d1b3ba3a36f86b65db60af37183d1/seed-...,/cluster/project/beltrao/jjaenes/26.01_batch-i...
3,001780d42f7d1b3ba3a36f86b65db60af37183d1,4,3,0.340523,0.08,0.0,0.29,0.35,"[0.09, 0.09, 0.08, 0.09, 0.07, 0.09, 0.07, 0.0...","[[0.79, 0.09, 0.08, 0.09, 0.08, 0.1, 0.07, 0.0...","[[0.76, 30.92, 31.12, 31.22, 30.98, 31.21, 31....","[0.79, 0.82, 0.41, 0.81, 0.76, 0.83, 0.5, 0.72...",001780d42f7d1b3ba3a36f86b65db60af37183d1/seed-...,/cluster/project/beltrao/jjaenes/26.01_batch-i...
4,001780d42f7d1b3ba3a36f86b65db60af37183d1,4,4,0.347117,0.08,0.0,0.29,0.36,"[0.09, 0.09, 0.08, 0.09, 0.07, 0.1, 0.07, 0.09...","[[0.79, 0.09, 0.07, 0.09, 0.08, 0.1, 0.07, 0.0...","[[0.76, 31.12, 31.44, 31.26, 31.04, 30.79, 30....","[0.79, 0.82, 0.4, 0.81, 0.76, 0.82, 0.51, 0.72...",001780d42f7d1b3ba3a36f86b65db60af37183d1/seed-...,/cluster/project/beltrao/jjaenes/26.01_batch-i...


In [3]:
pool_iptms = predictions.groupby('name').apply(pooled_ppi.predictions.agg_chain_pair_iptms, include_groups=False).reset_index()
pool_iptms['pool_id'] = [
    'p11484_p22943_p26570_p36110_p39954_p40008_p40022_p47128_p47822_p50276_q03941_q04304_q05543_q08230_q12181',
    'p07277_p14843_p15992_p29952_p30402_p38693_p38841_p40350_p53154_q00246_q03012_q06339_q12246',
]
pool_iptms

,name,chain_pair_iptm_best,chain_pair_iptm_mean,pool_id
0,000d99f7e4eba5e9ac87155974c36658f0058d61,"[[0.64, 0.08, 0.14, 0.1, 0.11, 0.09, 0.09, 0.0...","[[0.646, 0.08, 0.132, 0.1, 0.11000000000000001...",p11484_p22943_p26570_p36110_p39954_p40008_p400...
1,001780d42f7d1b3ba3a36f86b65db60af37183d1,"[[0.79, 0.09, 0.07, 0.09, 0.08, 0.1, 0.07, 0.0...","[[0.786, 0.09, 0.07400000000000001, 0.09, 0.08...",p07277_p14843_p15992_p29952_p30402_p38693_p388...


In [4]:
pair_iptms = pooled_ppi.predictions.explode_iptms(pool_iptms, columns_keep=['name'], columns_triu=['chain_pair_iptm_best', 'chain_pair_iptm_mean'])
pair_iptms

,ids,name,chain_pair_iptm_best,chain_pair_iptm_mean
0,"(p11484, p22943)",000d99f7e4eba5e9ac87155974c36658f0058d61,0.08,0.08
1,"(p11484, p26570)",000d99f7e4eba5e9ac87155974c36658f0058d61,0.14,0.132
2,"(p11484, p36110)",000d99f7e4eba5e9ac87155974c36658f0058d61,0.1,0.1
3,"(p11484, p39954)",000d99f7e4eba5e9ac87155974c36658f0058d61,0.11,0.11
4,"(p11484, p40008)",000d99f7e4eba5e9ac87155974c36658f0058d61,0.09,0.09
...,...,...,...,...
178,"(q00246, q06339)",001780d42f7d1b3ba3a36f86b65db60af37183d1,0.18,0.17
179,"(q00246, q12246)",001780d42f7d1b3ba3a36f86b65db60af37183d1,0.11,0.104
180,"(q03012, q06339)",001780d42f7d1b3ba3a36f86b65db60af37183d1,0.13,0.126
181,"(q03012, q12246)",001780d42f7d1b3ba3a36f86b65db60af37183d1,0.1,0.1
